### Text Classification using `BiRNN`
This notebook is just an expantion of the previous notebook. In this notebook we are going to apply `RNN` layers instead of `Conv1D` layers as we did in the last notebook. The reason being that RNN are better in processing sequencial data which is our case.

The reast of the notebook will remain the same as the previous notebook, where there's, we are only going to change few things.

### Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

import os, random, string, re

from tensorflow.keras.layers import TextVectorization

np.__version__

### Data
We are going to download the data (IMDB) dataset and load in google colab.

To download the data we are going to use the following command:

```shell
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

# extracting
!tar -xf aclImdb_v1.tar.gz
```

In [ ]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

The `aclImdb/train/pos` and `aclImdb/train/neg` folders contain text files, each of which represents one review (either `positive` or `negative`):

Let's check a single example of a positive review using the shell command `cat`.

In [ ]:
!cat aclImdb/train/pos/6248_7.txt

We are only interested in the `pos` and `neg` folders, so let's delete the other files

In [ ]:
!rm -r aclImdb/train/unsup
!rm -r aclImdb/test/unsup

Now our folder and file structures looks as follows:

```
?? acllmdb
  ?? test
    ?? neg
      ??...txt
    ?? pos
     ??...txt
  ?? train
    ?? neg
     ??...txt
    ?? pos
     ??...txt
```

We are going to use the [`tf.keras.preprocessing.text_dataset_from_directory`](https://keras.io/api/preprocessing/text#textdatasetfromdirectory-function) to generate a labeled [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) object from text files that are in our disk.

We are then going to generate three sets of data. The train, validation and test. The validation set will come as a fraction of `20%` from the train set.

### Loading the data using the `tf.keras.preprocessing.text_dataset_from_directory()`

In [ ]:
!rm -r aclImdb/train/.ipynb_checkpoints
!rm -r aclImdb/test/.ipynb_checkpoints

In [ ]:
BATCH_SIZE = 128 # we are going to increase the batchsize this time around to 128 from 32
SEED = 42

raw_train_ds = tf.keras.preprocessing.text_dataset_from_directory(
    "aclImdb/train",
    batch_size = BATCH_SIZE,
    validation_split = .2,
    subset = "training",
    seed = SEED
)
raw_valid_ds = tf.keras.preprocessing.text_dataset_from_directory(
    "aclImdb/train",
    batch_size = BATCH_SIZE,
    validation_split = .2,
    subset = "validation",
    seed = SEED
)
raw_test_ds = tf.keras.preprocessing.text_dataset_from_directory(
    "aclImdb/test",
    batch_size = BATCH_SIZE
)

Counting examples...

In [ ]:
print("train examples: %d"
 % tf.data.experimental.cardinality(raw_train_ds))
print("test examples: %d"
 % tf.data.experimental.cardinality(raw_test_ds))
print("valid examples: %d"
 % tf.data.experimental.cardinality(raw_valid_ds))

Let's check a some examples in a single batch.

In [ ]:
for text_batch, label_batch in raw_train_ds.take(1):
  for text, label in zip(text_batch, label_batch[:5]):
    print(f"text: {text}\nlabel: {label}\nn")

### Data cleaning
We are going to clean our data. By cleaning I mean removing unnessesarry text for example ``html`` tags. We are goiing to create a `standardization` function which will do the following:

1. lower case the text
2. remove html tags


In [ ]:
def standardization(input_data):
  input_data = tf.strings.lower(input_data)
  input_data = tf.strings.regex_replace(input_data, "<br/>", " ")
  return tf.strings.regex_replace(
      input_data, re.escape(string.punctuation), ""
  )

### Model Hyper Params

In [ ]:
MAX_FEATURES = 20000
EMBEDDING_DIM = 100
SEQUENCE_LENGTH = 500

### Text Vectorization
Now that we have our function that standadize text, we can then create a vectorization layer. WE are using this layer to:
1. split strings
2. map them to integer representations


In [ ]:
vectorize_layer = TextVectorization(
    standardize=standardization,
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH
)

In [ ]:
vectorize_layer.get_vocabulary()

As you can see that we only have the ``''`` and ``[UNK]`` as our vocabulary, we need to to make our `vectorize_layer` to adapt to our train features. So to do that we need to extract text features first from our `train_dataset` and the we call the `adapt`method to create the vocabulary.

In [ ]:
text_ds = raw_train_ds.map(lambda x, y: x)

In [ ]:
vectorize_layer.adapt(text_ds)

### Text vectorization layer.

There are two options we can use to vectorize our data.

1. **Make it part of the model**

This method allows us to pass the raw text strings to the model and the model will take care of the rest for us: The following example shows how we can use this method:

```py
input = tf.keras.Input(shape=(1,), dtype=tf.string, name='text')
x = vectorized_layer(input)
x = keras.layers.Embedding(MAX_FEATURES + 1, EMBEDDING_DIM)(x)

....
```

2. **Apply it to the text dataset**.

This method will obtain a dataset of word indices then feed those to the network.

> An important difference between the two is that option 2 enables you to do **asynchronous CPU processing and buffering** of your data when training on GPU. So if you're training the model on GPU, you probably want to go with this option to get the best performance



In [ ]:
def vectorize(text, label):
  text= tf.expand_dims(text, -1)
  return vectorize_layer(text), label

Vectorizing the data

In [ ]:
train_ds = raw_train_ds.map(vectorize)
valid_ds = raw_valid_ds.map(vectorize)
test_ds = raw_test_ds.map(vectorize)

### Async prefetching / buffering of the data 

In [ ]:
BUFFER_SIZE =  100
train_ds = train_ds.cache().prefetch(buffer_size=BUFFER_SIZE)
test_ds = test_ds.cache().prefetch(buffer_size=BUFFER_SIZE)
valid_ds = valid_ds.cache().prefetch(buffer_size=BUFFER_SIZE)

### Building the model


We are still going to make use of the functional API but this time around we will use the Bi-Direction RNN with LSTM layer as the forward layer and GRU as the backward layer.


In [ ]:
forward_layer = keras.layers.GRU(128, return_sequences=True, dropout=.5)
backward_layer = keras.layers.LSTM(128, activation='tanh', return_sequences=True,
                       go_backwards=True, dropout=.5)
inputs = keras.layers.Input(shape=(None, ), dtype="int64")

x = keras.layers.Embedding(MAX_FEATURES,EMBEDDING_DIM)(inputs)
x = keras.layers.Dropout(rate=.5)(x)

x = keras.layers.Bidirectional(
    forward_layer,
    backward_layer = backward_layer
)(x)

x = keras.layers.GRU(
    128, return_sequences=True
)(x)
x = keras.layers.LSTM(
    128, return_sequences=True,
   dropout=.5
)(x)
# flatten_layer = keras.layers.Flatten(name="flatten_layer")(lstm_layer)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(rate=.5)(x)

outputs = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs, name="rnn_model")

model.summary()


### Ploting the mode'l

In [ ]:
keras.utils.plot_model(model, dpi=64, show_shapes=True)

### Compiling the model

In [ ]:
model.compile(loss="binary_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])

### Training the model

In [ ]:
EPOCHS = 3

model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS)

### Evaluating the model

In [ ]:
model.evaluate(test_ds,verbose=1)

### Model inference

In the predict we are going to vectorize the text, then we are going to call the model.predict so that it will returns a prediction label to us.

In [ ]:
data = []
for text_batch, label_batch in raw_test_ds.take(1):
  for text, label in zip(text_batch, label_batch[:5]):
    data.append({"text":text, "label": label})

In [ ]:
def vectorize_text(text):
  text= tf.expand_dims(text, -1)
  return vectorize_layer(text)

vectorize_text("this movie sucks!")

In [ ]:
def make_prediction(text):
  vectors = vectorize_text(text)
  pred = tf.round(tf.squeeze(model(vectors)))
  return pred.numpy().astype("int32")

In [ ]:
print("real label\tpredicted label\tpredicted class")
for ele in data:
  predicted = make_prediction(ele["text"])
  print(f"{predicted}\t\t{ele['label']}\t\t{raw_train_ds.class_names[predicted]}")

### Making an End-End Model.

We are going to then make an end to end RNN model that will be able to process strings.

In [ ]:
forward_layer = keras.layers.GRU(128, return_sequences=True, dropout=.5)
backward_layer = keras.layers.LSTM(128, activation='tanh', return_sequences=True,
                       go_backwards=True, dropout=.5)
inputs = keras.layers.Input(shape=(1, ), dtype="string")
x = vectorize_layer(inputs)

x = keras.layers.Embedding(MAX_FEATURES,EMBEDDING_DIM)(x)
x = keras.layers.Dropout(rate=.5)(x)

x = keras.layers.Bidirectional(
    forward_layer,
    backward_layer = backward_layer
)(x)

x = keras.layers.GRU(
    128, return_sequences=True
)(x)
x = keras.layers.LSTM(
    128, return_sequences=True,
   dropout=.5
)(x)
# flatten_layer = keras.layers.Flatten(name="flatten_layer")(lstm_layer)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(rate=.5)(x)

outputs = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs, name="rnn_model")

model.summary()

### Training the end-to-end model

In [ ]:
model.compile(
    loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]
)

EPOCHS = 3

model.fit(raw_train_ds, validation_data=raw_valid_ds, epochs=EPOCHS)

### Evalutaing the model

In [ ]:
model.evaluate(raw_test_ds,verbose=1)

### Model Inference

In [ ]:
def make_prediction(text):
  text = tf.constant(text, dtype="string")
  text= tf.expand_dims(text, -1)
  pred = tf.round(tf.squeeze(model(text)))
  return pred.numpy().astype("int32")

In [ ]:
print("real label\tpredicted label\tpredicted class")
for ele in data:
  predicted = make_prediction(ele["text"])
  print(f"{predicted}\t\t{ele['label']}\t\t{raw_train_ds.class_names[predicted]}")